# Imports

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.builds import build_dir
from uavloc.data.tiles import ESRI_WORLD_IMAGERY, geo_mosaic
from uavloc.features.onnx_segmenter import OnnxSegmenter
from uavloc.geo import LatLon, haversine_m
from uavloc.planning.config import MissionInputs, PlannerConfig
from uavloc.planning.elevation import OpenMeteoElevation, check_elevation
from uavloc.planning.localizability import S1_CHANNEL, COVERAGE_CHANNEL, SelfSimilarity
from uavloc.planning.planner import plan
from uavloc.plotting.planning import plot_routes, show_raster
from uavloc.plotting.segmentation import colour_labels
from uavloc.track import REPO_ROOT

# Params

In [ ]:
SLUG = "route-planner"
START = (49.905, 23.885)        # west of Birky
END = (49.915, 23.925)          # east of Birky
ALTITUDE_M = 40.0
SPEED_MPS = 5.0
MAX_FLIGHT_TIME_S = 1200.0
HOVER_S = 0.0
SATELLITE_ONNX = "pocs/lraspp-pi5/models/lraspp-satellite.onnx"
TILE_CACHE = "data/derived/tile-cache"
PLANNER_CACHE = "data/derived/planner-cache"
DIAGNOSTIC_SPACING_M = 100.0    # coarser re-score, only for the s1 / coverage histograms

# Data

In [ ]:
figures = build_dir(SLUG, create=True) / "figures"
inputs = MissionInputs(LatLon(*START), LatLon(*END), ALTITUDE_M, SPEED_MPS, MAX_FLIGHT_TIME_S, HOVER_S)
config = PlannerConfig()
print(f"start-end {haversine_m(inputs.start, inputs.end):.0f} m")

# Model

In [ ]:
segmenter = OnnxSegmenter(REPO_ROOT / SATELLITE_ONNX)
localizability = SelfSimilarity(config.matcher, config.match_gsd_m, config.blind_s_max_m, config.blind_c_min)
print(segmenter.model_id, segmenter.tile_px, segmenter.n_classes)

# Run

In [ ]:
t0 = time.perf_counter()
result = plan(inputs, config, ESRI_WORLD_IMAGERY, segmenter, localizability,
              REPO_ROOT / TILE_CACHE, REPO_ROOT / PLANNER_CACHE)
plan_s = time.perf_counter() - t0
width_m, height_m = result.class_map.size_m
print(f"plan {plan_s:.0f} s; class map {result.class_map.pixels.shape} ({width_m:.0f} x {height_m:.0f} m); "
      f"grid {result.grid.shape}; blind limit {result.blind_limit_m:.0f} m; {len(result.routes)} routes")

In [ ]:
satellite = geo_mosaic(ESRI_WORLD_IMAGERY, result.area, config.zoom, REPO_ROOT / TILE_CACHE)
scores = localizability.scores(result.class_map, result.area, ALTITUDE_M, DIAGNOSTIC_SPACING_M)

In [ ]:
elevation = OpenMeteoElevation()
checks = [check_elevation(route.waypoints, ALTITUDE_M, elevation) for route in result.routes]

In [ ]:
def route_table(routes, checks) -> pd.DataFrame:
    return pd.DataFrame([{
        "weight": r.weight, "legs": len(r.waypoints) - 1, "length_m": round(r.length_m),
        "flight_min": round(r.flight_time_s / 60, 1), "over_time": r.exceeds_flight_time,
        "blind_share": round(r.blind_share, 3), "stretches": len(r.blind_stretches),
        "longest_blind_m": round(r.longest_blind_m), "over_blind_limit": r.exceeds_blind_limit,
        "ground_range_m": round(float(np.ptp(c.ground_m)), 1), "elevation_warnings": len(c.warnings),
    } for r, c in zip(routes, checks)])

In [ ]:
table = route_table(result.routes, checks)
table

In [ ]:
for route, check in zip(result.routes, checks):
    print(route.weight, check.warnings or "no elevation warnings")

# Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7), constrained_layout=True)
show_raster(axes[0], satellite)
axes[0].set_title("Esri z18")
show_raster(axes[1], result.class_map.with_pixels(colour_labels(result.class_map.pixels.argmax(-1))))
axes[1].set_title("class map (LR-ASPP satellite, ONNX)")
show_raster(axes[2], result.blind_map, cmap="RdYlGn_r", vmin=0, vmax=1)
axes[2].set_title(f"blind (s1 > {config.blind_s_max_m:g} m or coverage < {config.blind_c_min:g}), {ALTITUDE_M:g} m")
for ax in axes:
    plot_routes(ax, result.grid, result.routes)
axes[0].legend(loc="lower left", fontsize=8)
fig.savefig(figures / "routes-birky.png", dpi=130)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
s1 = scores.pixels[..., S1_CHANNEL].ravel()
axes[0].hist(np.clip(s1, 0, 30), bins=60)
axes[0].axvline(config.blind_s_max_m, color="red")
axes[0].set_xlabel("posterior long-axis std s1, m (clipped at 30)")
axes[1].hist(scores.pixels[..., COVERAGE_CHANNEL].ravel(), bins=60)
axes[1].axvline(config.blind_c_min, color="red")
axes[1].set_xlabel("weighted-class coverage")
fig.savefig(figures / "localizability-scores.png", dpi=130)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5), constrained_layout=True)
for route, check in zip(result.routes, checks):
    ax.plot(check.distances_m, check.height_above_ground_m, label=f"w={route.weight:g}")
ax.axhline(ALTITUDE_M, color="grey", ls="--")
ax.set_xlabel("distance along route, m")
ax.set_ylabel("height above ground, m")
ax.legend()
fig.savefig(figures / "elevation-birky.png", dpi=130)